# Calibration curve figures

Redraws the panels of Step 6 of `frc_calibration_checkerboard.ipynb` as individual, publication-
ready TIFFs, plus a plain-text notes file describing each one.

Four panels, each at both scales (full frame and 256 px patches), each written twice - once with
a legend and once without:

| panel | y | x | units |
|---|---|---|---|
| `ratio_vs_rco1_px` | $r_{co1}/r_{ref}$ | $r_{co1}$ | pixels |
| `ratio_vs_rco1_nm` | $r_{co1}/r_{ref}$ | $r_{co1}$ | nanometres |
| `rref_vs_rco1_px`  | $r_{ref}$ | $r_{co1}$ | pixels |
| `rref_vs_rco1_nm`  | $r_{ref}$ | $r_{co1}$ | nanometres |

Deliberately **not** drawn: the flagged-point rings, the identity line, the ratio = 1 line, and
the refit-without-flagged curve. The flagged points themselves are still plotted and are still
in the fit - only the crimson ring marking them is gone.

Nothing is recomputed. The measurements come from `calibration_points.csv` and the curve from the
parameters in `calibration_fits.csv`, both written by Step 7, so these figures cannot drift from
the run that produced them.

In [ ]:
# --- paths -----------------------------------------------------------------------------------
CALIBRATION_DIR = r"../outputs/checkerboard_calibration"
OUTPUT_DIR      = r"../outputs/calibration_curves"
NOTES_NAME      = "figure_notes.txt"

# --- what to write ---------------------------------------------------------------------------
PANELS_TO_SAVE = ["ratio_vs_rco1_px", "ratio_vs_rco1_nm", "rref_vs_rco1_px", "rref_vs_rco1_nm"]
SCALES_TO_SAVE = ["full_frame", "patch_256"]
LEGEND_VARIANTS = [True, False]      # each figure is written both ways

# In nanometres the single pixel-unit curve becomes one curve per magnification. False draws each
# across the whole fitted r_co1 range, which is what a pixel-unit calibration claims to be valid
# over regardless of magnification. True is the conservative choice - each curve stops where its
# own group's measurements stop - but with few points per group those segments shrink to slivers,
# and at full-frame scale they very nearly vanish.
CLIP_NM_CURVES_TO_GROUP = False

# --- file format -------------------------------------------------------------------------------
FORMATS          = ("tif",)          # e.g. ("tif", "png") to also get a quick-view copy
DPI              = 600
FIGSIZE          = (5.0, 4.4)
TIFF_COMPRESSION = "tiff_lzw"        # lossless; None writes an uncompressed TIFF

# --- reference lines ---------------------------------------------------------------------------
# Physical limits carried over from the calibration run, not free choices: the checkerboard halves
# sit on a lattice of twice the pixel spacing (4 px Nyquist period), and a two-image FRC cannot
# resolve below 2 px. In nanometres each limit becomes one line per magnification.
SHOW_LIMITS           = True
CHECKERBOARD_FLOOR_PX = 4.0
R_REF_LIMIT_PX        = 2.0

# --- styling ---------------------------------------------------------------------------------
SHOW_TITLES     = False
LABEL_FONTSIZE  = 12
TICK_FONTSIZE   = 10
LEGEND_FONTSIZE = 8
PREVIEW         = True               # draw the whole set inline at the end

In [ ]:
import os
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


def calibration_model(r, a, b, c, d):
    """Dumoux/Quoll calibration form: r_ref = a * exp(c * (r - b)) + d, in pixel units."""
    return a * np.exp(c * (np.asarray(r, dtype=np.float64) - b)) + d


points = pd.read_csv(os.path.join(CALIBRATION_DIR, "calibration_points.csv"))
fits = pd.read_csv(os.path.join(CALIBRATION_DIR, "calibration_fits.csv"))
usable = points[points["usable"]].copy()

PX_SIZES = sorted(usable["px_nm"].unique())
COLOURS = dict(zip(PX_SIZES, ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple"]))

SCALE_LABEL = {"full_frame": "full frame", "patch_256": "256 px patches"}


def get_fit(scale):
    """The `all` fit for one scale - the calibration itself, fitted to every usable point."""
    row = fits[(fits["scale"] == scale) & (fits["variant"] == "all")]
    return None if row.empty else row.iloc[0]


missing = [s for s in SCALES_TO_SAVE if s not in set(fits["scale"])]
if missing:
    raise ValueError(f"{missing} not in the fits file; it holds {sorted(set(fits['scale']))}")

print(f"read {len(points)} rows, {len(usable)} usable, from {CALIBRATION_DIR}")
print(f"pixel sizes: {', '.join(f'{p:.1f}' for p in PX_SIZES)} nm/px\n")
for scale in SCALES_TO_SAVE:
    u = usable[usable["scale"] == scale]
    f = get_fit(scale)
    print(f"{SCALE_LABEL[scale]:16s} n = {len(u):4d}   "
          f"({', '.join(f'{int(c)} at {p:.1f} nm/px' for p, c in u['px_nm'].value_counts().sort_index().items())})")
    print(f"{'':16s} fit: a = {f['a']:.4f}  b = {f['b']:.4f}  c = {f['c']:.4f}  d = {f['d']:.4f}"
          f"   RMSE = {f['rmse']:.3f} px   R2 = {f['r2']:.3f}")

In [ ]:
PANELS = {
    "ratio_vs_rco1_px": dict(
        x="r_co1_px", kind="ratio", units="px",
        xlabel="$r_{co1}$ (px)", ylabel="$r_{co1}\\ /\\ r_{ref}$",
        title="Calibration in pixel units", plain="ratio against r_co1, pixels",
        note=("How far the one-image estimate is off, plotted against the one-image value itself, "
              "in pixels.\n    A ratio of 1.4 means the one-image FRC reported a resolution 1.4x "
              "coarser than the gold standard.\n    This is the panel to read a correction factor "
              "off by eye.")),
    "ratio_vs_rco1_nm": dict(
        x="r_co1_nm", kind="ratio", units="nm",
        xlabel="$r_{co1}$ (nm)", ylabel="$r_{co1}\\ /\\ r_{ref}$",
        title="The same calibration in physical units", plain="ratio against r_co1, nanometres",
        note=("The same quantity against nanometres instead of pixels. One curve in pixel units "
              "becomes a family of\n    curves here, one per magnification, and that fanning-out "
              "is the argument for calibrating in pixels:\n    the bias follows the sampling grid, "
              "not the physical scale.")),
    "rref_vs_rco1_px": dict(
        x="r_co1_px", kind="rref", units="px",
        xlabel="$r_{co1}$ (px)", ylabel="$r_{ref}$ (px)",
        title="What the fit maps: $r_{ref} = f(r_{co1})$", plain="r_ref against r_co1, pixels",
        note=("The mapping the calibration actually performs: feed in a one-image resolution, read "
              "off the gold-standard\n    equivalent. Pixels. This is the fitted relation itself, "
              "not a derived ratio.")),
    "rref_vs_rco1_nm": dict(
        x="r_co1_nm", kind="rref", units="nm",
        xlabel="$r_{co1}$ (nm)", ylabel="$r_{ref}$ (nm)",
        title="The same mapping in physical units", plain="r_ref against r_co1, nanometres",
        note=("The same mapping in nanometres. Again one curve per magnification, since converting "
              "a pixel-unit\n    calibration to nanometres requires the pixel size.")),
}


def curve_segments(fit, kind, units, pts):
    """The fitted curve, ready to plot: (pixel size or None, x, y) for each segment.

    Only ever spans the r_co1 range that was measured - an exponential extrapolates violently, so
    a line drawn past the data would be an invention. In nanometres the single pixel-unit curve
    splits into one segment per magnification, clipped by default to the range its own group
    covers so the panel cannot imply coverage where nothing was measured.
    """
    xs = np.linspace(fit["x_min"], fit["x_max"], 400)
    ys = calibration_model(xs, fit["a"], fit["b"], fit["c"], fit["d"])
    ok = ys > 0
    xs, ys = xs[ok], ys[ok]
    if not len(xs):
        return []
    yy = xs / ys if kind == "ratio" else ys

    if units == "px":
        return [(None, xs, yy)]

    out = []
    for px, sub in pts.groupby("px_nm"):
        keep = ((xs >= sub["r_co1_px"].min()) & (xs <= sub["r_co1_px"].max())
                if CLIP_NM_CURVES_TO_GROUP else np.ones(xs.shape, dtype=bool))
        if keep.any():
            out.append((px, xs[keep] * px, yy[keep] if kind == "ratio" else yy[keep] * px))
    return out


def draw(ax, panel_key, scale, legend=True):
    """One panel onto one axes. Returns the number of points drawn."""
    p = PANELS[panel_key]
    pts = usable[usable["scale"] == scale]
    fit = get_fit(scale)
    big = scale == "full_frame"
    size = 85 if big else 14
    ycol = "ratio" if p["kind"] == "ratio" else ("r_ref_px" if p["units"] == "px" else "r_ref_nm")

    for px, sub in pts.groupby("px_nm"):
        ax.scatter(sub[p["x"]], sub[ycol], s=size, alpha=0.9 if big else 0.4,
                   color=COLOURS[px], edgecolors="black" if big else "none",
                   linewidths=0.6, zorder=3)

    segments = curve_segments(fit, p["kind"], p["units"], pts) if fit is not None else []
    for px, cx, cy in segments:
        ax.plot(cx, cy, color="black" if px is None else COLOURS[px],
                lw=2.2 if px is None else 1.8, zorder=5)

    if SHOW_LIMITS:
        if p["units"] == "px":
            ax.axvline(CHECKERBOARD_FLOOR_PX, color="grey", ls=":", lw=1.2, zorder=1)
            if p["kind"] == "rref":
                ax.axhline(R_REF_LIMIT_PX, color="grey", ls="-.", lw=1.0, zorder=1)
        else:
            for px in sorted(pts["px_nm"].unique()):
                ax.axvline(CHECKERBOARD_FLOOR_PX * px, color=COLOURS[px], ls=":", lw=1.0,
                           alpha=0.7, zorder=1)
                if p["kind"] == "rref":
                    ax.axhline(R_REF_LIMIT_PX * px, color=COLOURS[px], ls="-.", lw=0.9,
                               alpha=0.7, zorder=1)

    ax.set_xlabel(p["xlabel"], fontsize=LABEL_FONTSIZE)
    ax.set_ylabel(p["ylabel"], fontsize=LABEL_FONTSIZE)
    ax.tick_params(labelsize=TICK_FONTSIZE)
    ax.grid(alpha=0.25)
    if SHOW_TITLES:
        ax.set_title(p["title"], fontsize=10)

    # Legend handles are built by hand rather than from plot labels: the nm panels draw one curve
    # per magnification and one limit line per magnification, which auto-labelling would turn into
    # a legend longer than the data it describes.
    if legend:
        handles = [Line2D([], [], ls="none", marker="o", color=COLOURS[px],
                          markeredgecolor="black", markeredgewidth=0.6, markersize=7,
                          label=f"{px:.1f} nm/px")
                   for px in sorted(pts["px_nm"].unique())]
        if segments:
            handles.append(Line2D([], [], color="black" if p["units"] == "px" else "0.35",
                                  lw=2.0,
                                  label="fitted calibration" if p["units"] == "px"
                                  else "fitted calibration (one per pixel size)"))
        if SHOW_LIMITS:
            grey = "grey" if p["units"] == "px" else "0.35"
            suffix = "" if p["units"] == "px" else ", per pixel size"
            handles.append(Line2D([], [], color=grey, ls=":", lw=1.2,
                                  label=f"$r_{{co1}}$ sampling limit ({CHECKERBOARD_FLOOR_PX:.0f} px{suffix})"))
            if p["kind"] == "rref":
                handles.append(Line2D([], [], color=grey, ls="-.", lw=1.0,
                                      label=f"$r_{{ref}}$ limit ({R_REF_LIMIT_PX:.0f} px{suffix})"))
        ax.legend(handles=handles, fontsize=LEGEND_FONTSIZE, loc="best", framealpha=0.9)

    return len(pts)


def make_figure(panel_key, scale, legend):
    fig, ax = plt.subplots(figsize=FIGSIZE)
    draw(ax, panel_key, scale, legend)
    fig.tight_layout()
    return fig


print(f"{len(PANELS_TO_SAVE)} panels x {len(SCALES_TO_SAVE)} scales x "
      f"{len(LEGEND_VARIANTS)} legend variants x {len(FORMATS)} format(s) = "
      f"{len(PANELS_TO_SAVE) * len(SCALES_TO_SAVE) * len(LEGEND_VARIANTS) * len(FORMATS)} files")

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)


def save(fig, stem):
    written = []
    for ext in FORMATS:
        path = os.path.join(OUTPUT_DIR, f"{stem}.{ext}")
        kw = {}
        if ext.lower() in ("tif", "tiff") and TIFF_COMPRESSION:
            kw["pil_kwargs"] = {"compression": TIFF_COMPRESSION}
        fig.savefig(path, dpi=DPI, bbox_inches="tight", **kw)
        written.append(path)
    return written


manifest = []
for panel_key in PANELS_TO_SAVE:
    for scale in SCALES_TO_SAVE:
        for legend in LEGEND_VARIANTS:
            stem = f"{panel_key}_{scale}_{'legend' if legend else 'nolegend'}"
            fig = make_figure(panel_key, scale, legend)
            for path in save(fig, stem):
                manifest.append((os.path.basename(path), panel_key, scale, legend,
                                 os.path.getsize(path)))
            plt.close(fig)

print(f"wrote {len(manifest)} files to {OUTPUT_DIR}\n")
for name, _, _, _, nbytes in manifest:
    print(f"  {name:52s} {nbytes / 1e6:6.2f} MB")

In [ ]:
def fit_block(scale):
    f = get_fit(scale)
    u = usable[usable["scale"] == scale]
    return (f"  {SCALE_LABEL[scale]} ({scale})\n"
            f"    points fitted : {len(u)}  "
            f"({', '.join(f'{int(c)} at {p:.1f} nm/px' for p, c in u['px_nm'].value_counts().sort_index().items())})\n"
            f"    parameters    : a = {f['a']:.4f}   b = {f['b']:.4f}   "
            f"c = {f['c']:.4f}   d = {f['d']:.4f}\n"
            f"    quality       : RMSE = {f['rmse']:.3f} px, residual std = {f['resid_std']:.3f} px, "
            f"R2 = {f['r2']:.3f}\n"
            f"    drawn over    : r_co1 = {f['x_min']:.2f} - {f['x_max']:.2f} px "
            f"(the measured range; the curve is not extrapolated)\n")


colour_lines = "\n".join(f"    {COLOURS[px].replace('tab:', ''):8s} = {px:.1f} nm/px"
                         for px in PX_SIZES)
curve_note = ("each clipped to the r_co1 range its own group actually covers."
              if CLIP_NM_CURVES_TO_GROUP else
              "each spanning the whole fitted r_co1 range, converted at that pixel size.")
panel_lines = "\n\n".join(f"  {k}\n    {PANELS[k]['plain']}\n    {PANELS[k]['note']}"
                          for k in PANELS_TO_SAVE)
file_lines = "\n".join(f"  {n:52s} {PANELS[p]['plain']}, {SCALE_LABEL[s]}, "
                       f"{'with legend' if l else 'no legend'}"
                       for n, p, s, l, _ in manifest)

notes = f"""CALIBRATION CURVE FIGURES
{'=' * 78}
generated {datetime.now():%Y-%m-%d %H:%M} by calibration_curve_figure.ipynb
source    {CALIBRATION_DIR}
          calibration_points.csv (the measurements), calibration_fits.csv (the fitted curve)


THE TWO QUANTITIES
{'-' * 78}
  r_co1   resolution measured from a SINGLE image by splitting it into two checkerboard halves
          and running an FRC between them. Cheap, needs one exposure, and biased: the two halves
          sit on a lattice of twice the pixel spacing, so the estimate saturates against a 4 px
          floor and reports a resolution that is too coarse.

  r_ref   resolution from the two-image gold-standard FRC - two independent exposures of the same
          field, registered and correlated. The reference these figures calibrate against.

  ratio   r_co1 / r_ref. How far the one-image estimate is off. Above 1 means the one-image FRC
          reported a coarser (worse) resolution than the truth.

The calibration is the curve r_ref = a * exp(c * (r_co1 - b)) + d, fitted in PIXEL units. Pixels,
not nanometres, because the bias is geometric: it follows the sampling grid, so one curve serves
every magnification. The nanometre figures are that same curve converted per pixel size, which is
why they show a family of curves rather than one.


THE TWO SCALES
{'-' * 78}
  full frame      one point per field of view. Few points, each a whole-image measurement.
  256 px patches  one point per 256 px patch. Many more points, noisier individually, and they
                  show how much the ratio varies within a single field.

They are two separate calibrations, fitted independently, not one calibration shown twice.


WHAT IS ON THE PLOTS
{'-' * 78}
  filled circles  one measurement each, coloured by pixel size:
{colour_lines}
                  Full-frame points are large with a black outline; patch points are small and
                  semi-transparent, because there are hundreds of them.
                  Only measurements where BOTH the one-image and the gold-standard FRC crossed
                  the 0.143 threshold are plotted - a point with no crossing has no number.

  black curve     the fitted calibration (pixel-unit figures). In the nanometre figures this same
                  curve is drawn once per pixel size, in that pixel size's colour,
                  {curve_note}

  dotted vertical the r_co1 sampling limit, {CHECKERBOARD_FLOOR_PX:.0f} px. The checkerboard halves cannot resolve
                  anything finer; points near it are measurements in the regime the calibration
                  exists to correct. In nanometres this is {CHECKERBOARD_FLOOR_PX:.0f} px x the pixel size, so there is
                  one coloured line per magnification.

  dash-dot horiz. the r_ref limit, {R_REF_LIMIT_PX:.0f} px (r_ref figures only). A two-image FRC cannot report a
                  resolution finer than two pixels.


DELIBERATELY NOT SHOWN
{'-' * 78}
  flagged points  measurements whose FRC crossing fell in the last few rings are NOT ringed here.
                  They are still plotted and still in the fit - only the marker is omitted.
  identity line   the 1:1 line on the r_ref vs r_co1 figures.
  ratio = 1 line  the "no correction needed" line on the ratio figures.
  refit curve     the sensitivity-check curve refitted without the flagged points.
  See Step 6 and Step 8 of frc_calibration_checkerboard.ipynb for all four.


THE FITTED CURVES
{'-' * 78}
{''.join(fit_block(s) for s in SCALES_TO_SAVE)}

THE PANELS
{'-' * 78}
{panel_lines}


FILES
{'-' * 78}
Every figure is written twice: '_legend' has the key to the colours and lines drawn on it,
'_nolegend' is the same figure with nothing but the data, for a panel that gets its key from a
shared figure caption. {DPI} dpi, {FIGSIZE[0]:.1f} x {FIGSIZE[1]:.1f} in{', LZW-compressed' if TIFF_COMPRESSION else ''}.

{file_lines}
"""

notes_path = os.path.join(OUTPUT_DIR, NOTES_NAME)
with open(notes_path, "w", encoding="utf-8") as fh:
    fh.write(notes)

print(f"wrote {notes_path}\n")
print(notes)

In [ ]:
if PREVIEW:
    n_row, n_col = len(SCALES_TO_SAVE), len(PANELS_TO_SAVE)
    fig, axes = plt.subplots(n_row, n_col, figsize=(5.0 * n_col, 4.4 * n_row), squeeze=False)
    for r, scale in enumerate(SCALES_TO_SAVE):
        for c, panel_key in enumerate(PANELS_TO_SAVE):
            draw(axes[r, c], panel_key, scale, legend=True)
            axes[r, c].set_title(f"{panel_key}\n{SCALE_LABEL[scale]}", fontsize=10)
    fig.tight_layout()
    plt.show()